# Spark Implementation

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import (StringIndexer, RegexTokenizer, StopWordsRemover,
                                CountVectorizer, IDF, IndexToString)
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Get the private IP address for this instance
hostname_output = !hostname -I
master_private_ip = hostname_output[0].strip()

In [2]:
# Build spark session
spark = (
    SparkSession
      .builder
      .appName('ACLED_events')
      .master(f'spark://{master_private_ip}:7077')
      .config('spark.jars.packages', 'org.apache.hadoop:hadoop-aws:3.5.0')
      .getOrCreate()
)
spark

:: loading settings :: url = jar:file:/home/ubuntu/venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/ubuntu/.ivy2.5.2/cache
The jars for the packages stored in: /home/ubuntu/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-2cff93d2-bde9-49d9-9b3a-eb5df2ba4aaa;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.5.0 in central
	found software.amazon.awssdk#bundle;2.35.4 in central
	found software.amazon.s3.analyticsaccelerator#analyticsaccelerator-s3;1.3.1 in central
	found org.wildfly.openssl#wildfly-openssl;2.2.5.Final in central
:: resolution report :: resolve 553ms :: artifacts dl 26ms
	:: modules in use:
	org.apache.hadoop#hadoop-aws;3.5.0 from central in [default]
	org.wildfly.openssl#wildfly-openssl;2.2.5.Final from central in [default]
	software.amazon.awssdk#bundle;2.35.4 from central in [default]
	sof

In [3]:
# Look at number of partitions (w/3 spark worker nodes)
print(f"Available cores (minimum # partitions): {spark.sparkContext.defaultParallelism}")

Available cores (minimum # partitions): 6


In [5]:
# Add data to spark session
s3_uri = "s3a://dsan6000-data/acled_events.parquet"
acled_sdf = (
  spark
  .read
  .option('header', 'true')
  .parquet(s3_uri)
)
acled_sdf.printSchema()

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


root
 |-- event_id_cnty: string (nullable = true)
 |-- event_date: string (nullable = true)
 |-- year: long (nullable = true)
 |-- time_precision: long (nullable = true)
 |-- disorder_type: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- sub_event_type: string (nullable = true)
 |-- actor1: string (nullable = true)
 |-- assoc_actor_1: string (nullable = true)
 |-- inter1: string (nullable = true)
 |-- actor2: string (nullable = true)
 |-- assoc_actor_2: string (nullable = true)
 |-- inter2: string (nullable = true)
 |-- interaction: string (nullable = true)
 |-- civilian_targeting: string (nullable = true)
 |-- iso: long (nullable = true)
 |-- region: string (nullable = true)
 |-- country: string (nullable = true)
 |-- admin1: string (nullable = true)
 |-- admin2: string (nullable = true)
 |-- admin3: string (nullable = true)
 |-- location: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- geo_precisio

In [7]:
# Look at physical paritions in use
print(f"Available cores (minimum # partitions): {spark.sparkContext.defaultParallelism}")
print(f"ACLED_sdf # partitions: {acled_sdf.rdd.getNumPartitions()}")

Available cores (minimum # partitions): 6
ACLED_sdf # partitions: 6


In [ ]:
# Drop observations with missing text or event

text_sdf = (
    acled_sdf
    .select("event_date", "notes", "event_type") # TODO: add in other columns needed for the forecast model
    .where(F.col("notes").isNotNull() & (F.trim(F.col("notes")) != ""))
    .where(F.col("event_type").isNotNull())
)

In [10]:
# Check how many observations are left
text_sdf.count()

2781033

In [11]:
# Split into classifier training set, forecaster training set, and forecaster test set

# Make sure event_date is a date and add month column
sdf = (
    text_sdf
    .withColumn("event_date", F.to_date("event_date"))
    .withColumn("month", F.trunc("event_date", "month"))
)

# Count events per month
monthly = sdf.groupBy("month").count().orderBy("month").toPandas()
monthly["cum_share"] = monthly["count"].cumsum() / monthly["count"].sum()

# The first month where the cumulative share reaches 60% / 80%
cut_train = monthly.loc[monthly["cum_share"] >= 0.60, "month"].iloc[0]
cut_mid   = monthly.loc[monthly["cum_share"] >= 0.80, "month"].iloc[0]

# 60/40 Split for classifier training/test
clf_train_sdf = sdf.where(F.col("month") <= F.lit(cut_train))
fc_train_sdf  = sdf.where((F.col("month") > F.lit(cut_train)) & (F.col("month") <= F.lit(cut_mid)))
fc_test_sdf   = sdf.where(F.col("month") > F.lit(cut_mid))

/home/ubuntu/venv/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [12]:
# Check number of events in each split
for name, part in [("classifier train", clf_train_sdf),
                   ("forecaster train", fc_train_sdf),
                   ("forecaster test", fc_test_sdf)]:
    stats = part.agg(F.min("month"), F.max("month"),
                     F.countDistinct("month"), F.count("*")).first()
    print(f"{name:17} {stats[0]} to {stats[1]}  {stats[2]:>3} months  {stats[3]:>9,} events")

classifier train  1997-01-01 to 2022-11-01  311 months  1,692,206 events


forecaster train  2022-12-01 to 2024-05-01   18 months    541,727 events


forecaster test   2024-06-01 to 2025-09-01   16 months    547,100 events


In [ ]:
# Turn event labels into integers
label_indexer = StringIndexer(inputCol="event_type", outputCol="label", handleInvalid="keep")

# Tokenize, remove stop words
tokenizer = RegexTokenizer(inputCol="notes", outputCol="tokens", pattern="\\W+", minTokenLength=2)
stopwords = StopWordsRemover(inputCol="tokens", outputCol="words")

# Vectorize words
tf  = CountVectorizer(inputCol="words", outputCol="tf", vocabSize=50_000, minDF=5)
idf = IDF(inputCol="tf", outputCol="features")

# Build logistic regression model
clf = LogisticRegression(featuresCol="features", labelCol="label",
                         maxIter=100, regParam=0.01, family="multinomial")

pipeline = Pipeline(stages=[label_indexer, tokenizer, stopwords, tf, idf, clf])

# Fit the model
model = pipeline.fit(clf_train_sdf)

netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory


In [ ]:
# Turn integer labels back to original labels
labels = model.stages[0].labels
to_name = IndexToString(inputCol="prediction", outputCol="predicted_event_type", labels=labels)

# Predict on both forecast data sets
fc_train_pred = to_name.transform(model.transform(fc_train_sdf))
fc_test_pred  = to_name.transform(model.transform(fc_test_sdf))

fc_train_pred.select("event_date", "event_type", "predicted_event_type").show(5)

26/11/27 17:49:16 WARN DAGScheduler: Broadcasting large task binary with size 3.5 MiB
26/11/27 17:49:18 WARN DAGScheduler: Broadcasting large task binary with size 3.5 MiB


+----------+--------------------+--------------------+
|event_date|          event_type|predicted_event_type|
+----------+--------------------+--------------------+
|2022-12-01|            Protests|            Protests|
|2022-12-01|            Protests|            Protests|
|2022-12-01|Violence against ...|Violence against ...|
|2022-12-01|            Protests|            Protests|
|2022-12-01|            Protests|            Protests|
+----------+--------------------+--------------------+
only showing top 5 rows


In [ ]:
# Function for calculating macro-F1
def macro_f1(pred_sdf, n_classes):
    ev = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction",
                                           metricName="fMeasureByLabel")
    scores = [ev.setMetricLabel(float(i)).evaluate(pred_sdf) for i in range(n_classes)]
    return sum(scores) / n_classes, scores

# Calculate macro-F1
n = len(labels)
for name, pred in [("forecaster-train period", fc_train_pred), ("forecaster-test period", fc_test_pred)]:
    macro, per_class = macro_f1(pred, n)
    print(f"{name}: macro F1 = {macro:.3f}")
    for lab, s in zip(labels, per_class):
        print(f"    {lab:35} {s:.3f}")

26/11/27 17:49:31 WARN DAGScheduler: Broadcasting large task binary with size 3.5 MiB
26/11/27 17:49:59 WARN DAGScheduler: Broadcasting large task binary with size 3.5 MiB
26/11/27 17:50:27 WARN DAGScheduler: Broadcasting large task binary with size 3.5 MiB
26/11/27 17:50:55 WARN DAGScheduler: Broadcasting large task binary with size 3.5 MiB
26/11/27 17:51:22 WARN DAGScheduler: Broadcasting large task binary with size 3.5 MiB
26/11/27 17:51:49 WARN DAGScheduler: Broadcasting large task binary with size 3.5 MiB


forecaster-train period: macro F1 = 0.914
    Protests                            0.988
    Battles                             0.879
    Explosions/Remote violence          0.958
    Violence against civilians          0.888
    Riots                               0.863
    Strategic developments              0.907


26/11/27 17:52:13 WARN DAGScheduler: Broadcasting large task binary with size 3.5 MiB
26/11/27 17:52:49 WARN DAGScheduler: Broadcasting large task binary with size 3.5 MiB
26/11/27 17:53:24 WARN DAGScheduler: Broadcasting large task binary with size 3.5 MiB
26/11/27 17:53:59 WARN DAGScheduler: Broadcasting large task binary with size 3.5 MiB
26/11/27 17:54:31 WARN DAGScheduler: Broadcasting large task binary with size 3.5 MiB
26/11/27 17:55:10 WARN DAGScheduler: Broadcasting large task binary with size 3.5 MiB


forecaster-test period: macro F1 = 0.900
    Protests                            0.987
    Battles                             0.856
    Explosions/Remote violence          0.931
    Violence against civilians          0.883
    Riots                               0.855
    Strategic developments              0.889


In [16]:
# Check that predicted labels were added to forecast training set
fc_train_pred.columns

['event_date',
 'notes',
 'event_type',
 'month',
 'label',
 'tokens',
 'words',
 'tf',
 'features',
 'rawPrediction',
 'probability',
 'prediction',
 'predicted_event_type']

In [17]:
# Check that predicted labels were added to forecast test set
fc_test_pred.columns

['event_date',
 'notes',
 'event_type',
 'month',
 'label',
 'tokens',
 'words',
 'tf',
 'features',
 'rawPrediction',
 'probability',
 'prediction',
 'predicted_event_type']

Next Steps: Reference TODO text above in filtering columns cell. Additional features should be added to this line for the forecast model. It won't affect the nlp steps.